# Cruise Control over Hills — Python / `python-control`

Same plant as the JS playground, done rigorously. Runs entirely in your browser
via Pyodide. First cell installs `python-control` (a few seconds).

In [ ]:
%pip install -q control
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

In [ ]:
m, c, g = 1200.0, 40.0, 9.81      # mass, drag coeff, gravity
umax, vref = 8000.0, 20.0          # actuator limit, target speed
Kp, Ki, Kd = 800.0, 120.0, 60.0    # <-- students tune these

def slope(s):                      # road grade dh/ds -> gravity disturbance
    return (6/50)*np.cos(s/50) + (3/23)*np.cos(s/23 + 1)

def rhs(t, x, mem):
    s, v = x
    e = vref - v
    mem['I'] += e * (t - mem['t']); mem['t'] = t
    de = (e - mem['e']) / max(t - mem['tp'], 1e-6) if t > mem['tp'] else 0.0
    mem['e'], mem['tp'] = e, t
    u = np.clip(Kp*e + Ki*mem['I'] + Kd*de, -umax, umax)
    theta = np.arctan(slope(s))
    return [v, (u - m*g*np.sin(theta) - c*v)/m]

mem = {'I':0.0, 'e':0.0, 't':0.0, 'tp':0.0}
sol = solve_ivp(rhs, (0, 60), [0, 0], args=(mem,), max_step=0.05, dense_output=True)
t = np.linspace(0, 60, 1000); v = sol.sol(t)[1]

plt.figure(figsize=(8,3))
plt.axhline(vref, ls='--', label='target'); plt.plot(t, v, label='measured')
plt.xlabel('time (s)'); plt.ylabel('speed (m/s)'); plt.legend(); plt.grid(alpha=.3)
plt.title('Cruise control tracking'); plt.show()

**Exercise.** Retune the gains to keep peak speed error under 1 m/s over the hills.
Then linearize the plant, build the transfer function with `control.tf`, and compare
the Bode / step response to what you observe above.